Exploratory Data Analysis on Indonesia Stunting Dataset
*Co-authored with CoCo*

## 1.Load & Inspect Data

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load the stunting dataset
df = pd.read_csv('master_dataset_stunting_10000_english.csv')
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nColumn names:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

Dataset shape: 10,000 rows x 33 columns

Column names:
   1. Child_ID
   2. National_ID_NIK
   3. Child_Name
   4. Parent_Name
   5. Gender
   6. Neighborhood_RT_RW
   7. Birth_Date
   8. Measurement_Date
   9. Age_at_Measurement_Months
  10. Mother_Height_cm
  11. Mother_Education
  12. Mother_Pregnancy_Age
  13. Mother_Chronic_Energy_Deficiency
  14. Mother_ANC_Visits
  15. Mother_Iron_Supplementation
  16. Birth_Spacing_Months
  17. Birth_Weight_kg
  18. Birth_Length_cm
  19. Weight_kg
  20. Height_or_Length_cm
  21. Head_Circumference_cm
  22. Measurement_Posture
  23. Exclusive_Breastfeeding
  24. Complementary_Feeding_Animal_Protein
  25. Infection_History
  26. Immunization_Status
  27. Clean_Water_Access
  28. Latrine_Ownership
  29. Family_Smoking_Habit
  30. Social_Assistance_Recipient
  31. Health_Insurance_Status
  32. Data_Source
  33. Monitoring_Type


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 33 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Child_ID                              10000 non-null  object 
 1   National_ID_NIK                       10000 non-null  int64  
 2   Child_Name                            10000 non-null  object 
 3   Parent_Name                           10000 non-null  object 
 4   Gender                                10000 non-null  object 
 5   Neighborhood_RT_RW                    10000 non-null  object 
 6   Birth_Date                            10000 non-null  object 
 7   Measurement_Date                      10000 non-null  object 
 8   Age_at_Measurement_Months             10000 non-null  float64
 9   Mother_Height_cm                      10000 non-null  float64
 10  Mother_Education                      10000 non-null  object 
 11  Mother_Pregnancy

> 💡 **Insight:** Datanya ada 10.000 baris dan 33 kolom — jumlah kolom yang banyak ini karena dataset dummy ini sengaja dilengkapi fitur tambahan (pendidikan ibu, ASI eksklusif, akses air bersih, dll) yang nggak ada di data asli Yayasan, buat latihan analisis yang lebih kaya.

In [ ]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, '%': missing_pct})
missing_df = missing_df[missing_df['Missing'] > 0].sort_values('%', ascending=False)

if len(missing_df) > 0:
    print("Columns with missing values:")
    display(missing_df)
else:
    print("No missing values in the dataset.")

print(f"\nFirst 5 rows:")
display(df.head())

Columns with missing values:


,Missing,%
Infection_History,6003,60.03



First 5 rows:


,Child_ID,National_ID_NIK,Child_Name,Parent_Name,Gender,Neighborhood_RT_RW,Birth_Date,Measurement_Date,Age_at_Measurement_Months,Mother_Height_cm,...,Complementary_Feeding_Animal_Protein,Infection_History,Immunization_Status,Clean_Water_Access,Latrine_Ownership,Family_Smoking_Habit,Social_Assistance_Recipient,Health_Insurance_Status,Data_Source,Monitoring_Type
0,CHILD-00001,3576664279485493,Child 1,Parent 1,Female,006/002,2021-12-31,2026-04-06,51.15,149.5,...,Inadequate,NaN,Complete,Adequate,With_Septic_Tank,No,No,Active,Posyandu,Routine_Monitoring
1,CHILD-00002,3573855171250169,Child 2,Parent 2,Female,001/004,2021-12-27,2026-06-22,53.82,151.4,...,Inadequate,Diarrhea,Incomplete,Inadequate,With_Septic_Tank,Yes,No,Active,Puskesmas,Routine_Monitoring
2,CHILD-00003,3572198942856425,Child 3,Parent 3,Male,003/003,2023-01-09,2026-06-04,40.80,161.1,...,Inadequate,ARI,Complete,Adequate,With_Septic_Tank,No,Yes,Uninsured,Posyandu,Routine_Monitoring
3,CHILD-00004,3579179708665308,Child 4,Parent 4,Male,004/001,2025-07-04,2026-02-06,7.13,150.7,...,Inadequate,Diarrhea,Not_Yet_Due,Inadequate,Without_Septic_Tank,No,No,Active,Puskesmas,Routine_Monitoring
4,CHILD-00005,3575165270620543,Child 5,Parent 5,Male,008/001,2023-04-26,2026-02-16,33.74,154.4,...,Adequate,NaN,Complete,Adequate,With_Septic_Tank,Yes,Yes,Uninsured,Posyandu,Routine_Monitoring


> 💡 **Insight:** Tahap ini mencari kolom mana saja yang datanya bolong (kosong/NaN). Dari hasilnya, cuma kolom 'Infection_History' (riwayat infeksi) yang banyak kosong, sehingga perlu diperiksa dan ditentukan penanganannya sebelum analisis lebih lanjut.

In [ ]:
# Check for duplicate rows
duplicate_rows = df.duplicated().sum()
if duplicate_rows > 0:
    print(f"Number of duplicate rows: {duplicate_rows}")
else:
    print("No duplicate rows found.")

No duplicate rows found.


> 💡 **Insight:** Tahap ini cek ada baris yang kembar persis (data ke-input dua kali) atau nggak. Hasilnya bersih, tidak ada duplikat — artinya tiap baris memang mewakili satu anak yang beda.

## 2.Numeric Distributions

In [ ]:
# Descriptive statistics for numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns ({len(numeric_cols)}):")
display(df[numeric_cols].describe().round(2))

Numeric columns (11):


,National_ID_NIK,Age_at_Measurement_Months,Mother_Height_cm,Mother_Pregnancy_Age,Mother_ANC_Visits,Birth_Spacing_Months,Birth_Weight_kg,Birth_Length_cm,Weight_kg,Height_or_Length_cm,Head_Circumference_cm
count,1.000000e+04,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00
mean,3.575446e+15,29.61,152.89,25.99,5.01,24.76,3.14,49.50,12.51,91.46,46.96
std,2.601714e+12,17.07,4.99,4.00,2.58,19.99,0.55,2.14,4.22,66.43,5.01
min,3.571001e+15,0.33,134.40,12.00,1.00,0.00,1.50,40.00,0.00,0.00,0.00
25%,3.573172e+15,14.61,149.50,23.00,3.00,0.00,2.80,48.70,9.70,77.60,46.30
50%,3.575407e+15,29.54,152.90,26.00,5.00,25.00,3.20,49.80,13.20,90.40,48.20
75%,3.577700e+15,44.45,156.20,29.00,7.00,42.00,3.60,50.90,15.90,99.62,49.60
max,3.579999e+15,59.10,172.20,41.00,9.00,59.00,4.00,52.00,20.70,999.00,54.60


> 💡 **Insight:** Ini ringkasan statistik dasar (rata-rata, minimum, maksimum, dst) buat semua kolom angka sekaligus. Gunanya buat 'sniff test' cepat — kalau ada angka minimum atau maksimum yang kelihatan aneh (misal tinggi badan minus atau 999), itu tanda ada data kotor yang perlu dibersihin sebelum lanjut analisis.

## 3.Data Cleaning and Preparation

### Mengatasi Nilai 999 dan 0 Pada Data

In [ ]:
# Melihat data yang nilainya 0 atau 999
extreme_anthro = df[
    (df['Weight_kg'].isin([0, 999])) |
    (df['Height_or_Length_cm'].isin([0, 999])) |
    (df['Head_Circumference_cm'].isin([0, 999])) |
    (df['Birth_Weight_kg'].isin([0, 999])) |
    (df['Birth_Length_cm'].isin([0, 999]))
]

display(
    extreme_anthro[
        ['Child_ID',
         'Age_at_Measurement_Months',
         'Weight_kg',
         'Height_or_Length_cm',
         'Head_Circumference_cm',
         'Birth_Weight_kg',
         'Birth_Length_cm']
    ]
)

,Child_ID,Age_at_Measurement_Months,Weight_kg,Height_or_Length_cm,Head_Circumference_cm,Birth_Weight_kg,Birth_Length_cm
47,CHILD-00048,43.27,15.3,999.0,51.8,2.6,48.6
106,CHILD-00107,51.22,0.0,110.6,50.7,3.3,50.0
116,CHILD-00117,8.02,6.3,999.0,42.4,3.5,48.1
120,CHILD-00121,17.12,0.0,79.1,46.6,2.8,48.5
212,CHILD-00213,37.82,14.5,999.0,47.0,3.9,48.3
...,...,...,...,...,...,...,...
9706,CHILD-09707,45.93,16.3,0.0,48.6,3.2,49.5
9721,CHILD-09722,51.38,15.0,999.0,48.4,3.6,49.8
9832,CHILD-09833,15.64,10.5,80.3,0.0,3.9,51.2
9937,CHILD-09938,11.43,8.7,999.0,46.5,3.5,50.4


> 💡 **Insight:** Ada 200 data anak mencakup berat badan, tinggi/panjang badan, lingkar kepala, berat badan lahir, dan panjang badan lahir yang nilainya 0 dan 999. Tahap selanjutnya data tersebut akan diubah menjadi NaN.

In [ ]:
# Mengatasi nilai 0 dan 999 pada data

anthro_cols = [
    'Weight_kg',
    'Height_or_Length_cm',
    'Head_Circumference_cm',
    'Birth_Weight_kg',
    'Birth_Length_cm'
]

# Mengubah nilai 0 dan 999 menjadi NaN
df[anthro_cols] = df[anthro_cols].replace([0, 999], np.nan)

# Menampilkan jumlah missing setelah cleaning
print("Jumlah missing setelah penanganan nilai 0 dan 999:")
display(df[anthro_cols].isna().sum())

Jumlah missing setelah penanganan nilai 0 dan 999:


,0
Weight_kg,50
Height_or_Length_cm,100
Head_Circumference_cm,50
Birth_Weight_kg,0
Birth_Length_cm,0


### Data Type Conversion

In [ ]:
import pandas as pd

# Mengkonversi kolom tanggal ke datetime
df['Birth_Date'] = pd.to_datetime(df['Birth_Date'])
df['Measurement_Date'] = pd.to_datetime(df['Measurement_Date'])

# Ubah Mother_Education menjadi tipe kategorikal berurutan.
education_order = ['Primary_or_Lower', 'Middle_School', 'High_School', 'Higher_Education']
df['Mother_Education'] = pd.Categorical(df['Mother_Education'], categories=education_order, ordered=True)

# Konversikan kolom kategorikal relevan lainnya ke tipe data 'category' untuk optimalisasi memori
for col in ['Gender', 'Mother_Chronic_Energy_Deficiency', 'Mother_Iron_Supplementation',
            'Measurement_Posture', 'Exclusive_Breastfeeding', 'Complementary_Feeding_Animal_Protein',
            'Immunization_Status', 'Clean_Water_Access', 'Latrine_Ownership', 'Family_Smoking_Habit',
            'Social_Assistance_Recipient', 'Health_Insurance_Status', 'Data_Source', 'Monitoring_Type']:
    if col in df.columns:
        df[col] = df[col].astype('category')

### Mengatasi Missing Values


In [ ]:
# Nilai NA pada 'Infection_History' diisi dengan 'No_Record'
df['Infection_History'] = df['Infection_History'].fillna('No_Record').astype('category')

print("Missing values in 'Infection_History' handled.")

Missing values in 'Infection_History' handled.


### Cek Perubahan Setelah Data Cleaning

In [ ]:
print("DataFrame Info after Cleaning:")
print(df.info())

DataFrame Info after Cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 33 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   Child_ID                              10000 non-null  object        
 1   National_ID_NIK                       10000 non-null  int64         
 2   Child_Name                            10000 non-null  object        
 3   Parent_Name                           10000 non-null  object        
 4   Gender                                10000 non-null  category      
 5   Neighborhood_RT_RW                    10000 non-null  object        
 6   Birth_Date                            10000 non-null  datetime64[ns]
 7   Measurement_Date                      10000 non-null  datetime64[ns]
 8   Age_at_Measurement_Months             10000 non-null  float64       
 9   Mother_Height_cm                      1000

# 4.EDA

In [ ]:
# Key health metrics distributions
fig = make_subplots(rows=2, cols=3, subplot_titles=[
    'Age at Measurement (months)', 'Birth Weight (kg)', 'Birth Length (cm)',
    'Weight (kg)', 'Height/Length (cm)', 'Head Circumference (cm)'])

fig.add_trace(go.Histogram(x=df['Age_at_Measurement_Months'], nbinsx=30, marker_color='#636EFA'), row=1, col=1)
fig.add_trace(go.Histogram(x=df['Birth_Weight_kg'], nbinsx=30, marker_color='#EF553B'), row=1, col=2)
fig.add_trace(go.Histogram(x=df['Birth_Length_cm'], nbinsx=30, marker_color='#00CC96'), row=1, col=3)
fig.add_trace(go.Histogram(x=df['Weight_kg'], nbinsx=30, marker_color='#AB63FA'), row=2, col=1)
fig.add_trace(go.Histogram(x=df['Height_or_Length_cm'], nbinsx=30, marker_color='#FFA15A'), row=2, col=2)
fig.add_trace(go.Histogram(x=df['Head_Circumference_cm'], nbinsx=30, marker_color='#19D3F3'), row=2, col=3)

fig.update_layout(height=500, title_text='Child Health Metrics Distributions', showlegend=False)
fig.show()

> 💡 **Insight:** Enam histogram ini nunjukkin sebaran usia, berat lahir, panjang lahir, berat sekarang, tinggi sekarang, dan lingkar kepala anak. Usia anak tersebar cukup merata dari 0-59 bulan (bagus, artinya sampelnya nggak nge-cluster di satu kelompok usia doang). Berat lahir kebanyakan di sekitar 3,2 kg (normal), tapi ada beberapa anak di bawah 2,5 kg (kategori berat lahir rendah, salah satu faktor risiko stunting yang dikenal).  Visualisasi ini digunakan sebagai gambaran awal distribusi variabel antropometri sebelum dilakukan analisis status gizi berdasarkan Z-score WHO.

In [ ]:
# Mother's health metrics
fig = make_subplots(rows=1, cols=3, subplot_titles=[
    'Mother Height (cm)', 'Mother Pregnancy Age', 'Mother ANC Visits'])

fig.add_trace(go.Histogram(x=df['Mother_Height_cm'], nbinsx=30, marker_color='#FF6692'), row=1, col=1)
fig.add_trace(go.Histogram(x=df['Mother_Pregnancy_Age'], nbinsx=20, marker_color='#B6E880'), row=1, col=2)
fig.add_trace(go.Histogram(x=df['Mother_ANC_Visits'], nbinsx=15, marker_color='#FECB52'), row=1, col=3)

fig.update_layout(height=350, title_text="Mother's Health Indicators", showlegend=False)
fig.show()

> 💡 **Insight:** Tiga histogram ini soal kondisi ibu: tinggi badan, usia saat hamil, dan jumlah kunjungan pemeriksaan kehamilan (ANC). Tinggi ibu rata-rata sekitar 153 cm, sedangkan usia saat hamil paling banyak berada pada kisaran pertengahan 20-an hingga awal 30-an tahun. Jumlah kunjungan ANC tersebar pada rentang 1-9 kali dengan distribusi yang relatif merata. Visualisasi ini memberikan gambaran awal mengenai karakteristik ibu sebelum dilakukan analisis hubungan dengan status gizi anak.

## Categorical Analysis

In [ ]:
# Gender distribution
categorical_cols = df.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")

fig = make_subplots(rows=2, cols=3, specs=[[{'type':'domain'}]*3, [{'type':'domain'}]*3],
                    subplot_titles=['Gender', 'Mother Education', 'Mother CED',
                                   'Exclusive Breastfeeding', 'Immunization Status', 'Infection History'])

for i, col in enumerate(['Gender', 'Mother_Education', 'Mother_Chronic_Energy_Deficiency',
                         'Exclusive_Breastfeeding', 'Immunization_Status', 'Infection_History']):
    counts = df[col].value_counts()
    row = i // 3 + 1
    col_idx = i % 3 + 1
    fig.add_trace(go.Pie(labels=counts.index, values=counts.values, hole=0.4,
                         textinfo='label+percent', textposition='inside'), row=row, col=col_idx)

fig.update_layout(height=600, title_text='Categorical Feature Distributions', showlegend=False)
fig.show()

Categorical columns (4): ['Child_ID', 'Child_Name', 'Parent_Name', 'Neighborhood_RT_RW']


> 💡 **Insight:** Donut chart ini nunjukkin proporsi kategori-kategori penting: gender (hampir seimbang, 50.6% perempuan vs 49.4% laki-laki), pendidikan ibu, status kekurangan energi kronis (KEK) pada ibu, ASI eksklusif, status imunisasi, dan riwayat infeksi.

In [ ]:
# WASH & socioeconomic factors
fig = make_subplots(rows=2, cols=3, specs=[[{'type':'domain'}]*3, [{'type':'domain'}]*3],
                    subplot_titles=['Clean Water Access', 'Latrine Ownership', 'Family Smoking',
                                   'Social Assistance', 'Health Insurance', 'Complementary Feeding'])

for i, col in enumerate(['Clean_Water_Access', 'Latrine_Ownership', 'Family_Smoking_Habit',
                         'Social_Assistance_Recipient', 'Health_Insurance_Status',
                         'Complementary_Feeding_Animal_Protein']):
    counts = df[col].value_counts()
    row = i // 3 + 1
    col_idx = i % 3 + 1
    fig.add_trace(go.Pie(labels=counts.index, values=counts.values, hole=0.4,
                         textinfo='label+percent', textposition='inside'), row=row, col=col_idx)

fig.update_layout(height=600, title_text='WASH & Socioeconomic Factors', showlegend=False)
fig.show()

> 💡 **Insight:** Ini donut chart buat faktor lingkungan (akses air bersih, kepemilikan jamban sehat, kebiasaan merokok keluarga) dan faktor sosial-ekonomi (penerima bansos, kepesertaan asuransi kesehatan). Faktor-faktor ini disebut WASH (Water, Sanitation, Hygiene) dan dikenal luas berhubungan dengan risiko infeksi berulang, yang pada akhirnya bisa mengganggu penyerapan gizi anak.

## Stunting Risk Analysis

In [ ]:
# Exploratory analysis of Height/Length by age group and gender
# This analysis uses raw Height/Length measurements,
# not WHO Height-for-Age Z-scores.

# Create age groups
df['Age_Group'] = pd.cut(df['Age_at_Measurement_Months'],
                         bins=[-1, 6, 12, 24, 36, 48, 60],
                         labels=['0-6m', '7-12m', '13-24m', '25-36m', '37-48m', '49-60m'])

# Height by age group and gender
fig = px.box(df, x='Age_Group', y='Height_or_Length_cm', color='Gender',
             title='Height/Length by Age Group and Gender',
             labels={'Height_or_Length_cm': 'Height/Length (cm)', 'Age_Group': 'Age Group'},
             color_discrete_sequence=['#FF6B6B', '#4ECDC4'])
fig.update_layout(height=450)
fig.show()

# Summary stats by age group
print("\nMedian height by age group and gender:")
display(df.groupby(['Age_Group', 'Gender'])['Height_or_Length_cm'].median().unstack().round(1))


Median height by age group and gender:


/tmp/ipykernel_1534/2678893232.py:20: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



Gender,Female,Male
Age_Group,,
0-6m,56.8,56.8
7-12m,69.0,69.1
13-24m,81.2,81.3
25-36m,90.8,90.9
37-48m,98.0,98.1
49-60m,104.7,105.1


> 💡 **Insight:** Grafik ini bandingin tinggi badan anak per kelompok usia (0-6 bulan, 7-12 bulan, dst), dipisah per gender. Distribusi tinggi/panjang badan menunjukkan kecenderungan median yang meningkat seiring bertambahnya kelompok usia, baik pada anak laki-laki maupun perempuan. Catatan: ini masih perbandingan tinggi badan mentah berdasarkan kelompok usia, BUKAN Z-score resmi WHO. Perhitungan Z-score yang sesungguhnya baru dilakukan ditahap berikutnya.

In [ ]:
# Risk factors analysis: Birth weight categories
df['Low_Birth_Weight'] = df['Birth_Weight_kg'] < 2.5

# Compare height between low and normal birth weight
fig = px.box(df, x='Low_Birth_Weight', y='Height_or_Length_cm', color='Low_Birth_Weight',
             title='Height/Length: Low Birth Weight vs Normal',
             labels={'Low_Birth_Weight': 'Low Birth Weight (<2.5kg)', 'Height_or_Length_cm': 'Height/Length (cm)'},
             color_discrete_sequence=['#4ECDC4', '#FF6B6B'])
fig.update_layout(height=400)
fig.show()

lbw_count = df['Low_Birth_Weight'].sum()
print(f"\nLow birth weight (<2.5kg): {lbw_count:,} children ({lbw_count/len(df)*100:.1f}%)")
print(f"Normal birth weight: {len(df)-lbw_count:,} children ({(len(df)-lbw_count)/len(df)*100:.1f}%)")


Low birth weight (<2.5kg): 787 children (7.9%)
Normal birth weight: 9,213 children (92.1%)


> 💡 **Insight:**  Boxplot ini membandingkan distribusi tinggi/panjang badan anak berdasarkan kategori berat lahir. Terlihat perbedaan distribusi antara kelompok berat lahir rendah (<2,5 kg) dan kelompok lainnya. Namun, karena yang dibandingkan masih tinggi/panjang badan mentah dan belum mempertimbangkan usia maupun Z-score TB/U, grafik ini belum dapat digunakan untuk menyimpulkan status stunting atau membuktikan berat lahir rendah sebagai faktor risiko stunting. Analisis hubungan dengan stunting dilakukan lebih lanjut menggunakan status gizi berdasarkan Z-score WHO

In [ ]:
# Mother's factors vs child height
fig = make_subplots(rows=1, cols=2, subplot_titles=['By Mother Education', 'By Mother CED Status'])

for i, edu in enumerate(df['Mother_Education'].unique()):
    subset = df[df['Mother_Education'] == edu]['Height_or_Length_cm']
    fig.add_trace(go.Box(y=subset, name=edu), row=1, col=1)

for i, ced in enumerate(df['Mother_Chronic_Energy_Deficiency'].unique()):
    subset = df[df['Mother_Chronic_Energy_Deficiency'] == ced]['Height_or_Length_cm']
    fig.add_trace(go.Box(y=subset, name=f"CED: {ced}"), row=1, col=2)

fig.update_layout(height=450, title_text="Child Height by Mother's Factors", showlegend=False)
fig.update_yaxes(title_text='Height/Length (cm)')
fig.show()

> 💡 **Insight:** Grafik ini menunjukkan perbandingan distribusi tinggi/panjang badan anak berdasarkan pendidikan dan status KEK ibu. Perbedaan yang terlihat masih bersifat deskriptif karena menggunakan tinggi/panjang badan mentah. Oleh karena itu, grafik ini belum dapat menunjukkan apakah karakteristik ibu tersebut berhubungan dengan stunting. Hubungan dengan stunting akan dianalisis setelah status stunting ditentukan berdasarkan Z-score TB/U WHO.

In [ ]:
# Breastfeeding and complementary feeding impact
fig = make_subplots(rows=1, cols=2, subplot_titles=['By Exclusive Breastfeeding', 'By Complementary Feeding'])

for bf in df['Exclusive_Breastfeeding'].unique():
    subset = df[df['Exclusive_Breastfeeding'] == bf]['Height_or_Length_cm']
    fig.add_trace(go.Box(y=subset, name=f"BF: {bf}"), row=1, col=1)

for cf in df['Complementary_Feeding_Animal_Protein'].unique():
    subset = df[df['Complementary_Feeding_Animal_Protein'] == cf]['Height_or_Length_cm']
    fig.add_trace(go.Box(y=subset, name=cf), row=1, col=2)

fig.update_layout(height=450, title_text='Feeding Practices vs Child Height', showlegend=False)
fig.update_yaxes(title_text='Height/Length (cm)')
fig.show()

> 💡 **Insight:** Ini ngebandingin tinggi badan anak berdasarkan riwayat ASI eksklusif dan kecukupan asupan protein hewani sebagai makanan pendamping ASI. WHO merekomendasikan ASI eksklusif 6 bulan pertama karena dianggap memberi nutrisi dan perlindungan imun paling optimal di masa krusial. Namun, hasil ini belum dapat digunakan untuk menyimpulkan adanya hubungan atau pengaruh praktik pemberian makan terhadap stunting, karena tinggi/panjang badan yang digunakan masih merupakan pengukuran mentah. Analisis hubungan dengan status stunting dilakukan setelah status gizi ditentukan berdasarkan Z-score TB/U WHO.

In [ ]:
# WASH factors impact
fig = make_subplots(rows=1, cols=3, subplot_titles=['Clean Water', 'Latrine', 'Family Smoking'])

for val in df['Clean_Water_Access'].dropna().unique():
    subset = df[df['Clean_Water_Access'] == val]['Height_or_Length_cm']
    fig.add_trace(go.Box(y=subset, name=val), row=1, col=1)

for val in df['Latrine_Ownership'].dropna().unique():
    subset = df[df['Latrine_Ownership'] == val]['Height_or_Length_cm']
    fig.add_trace(go.Box(y=subset, name=val), row=1, col=2)

for val in df['Family_Smoking_Habit'].dropna().unique():
    subset = df[df['Family_Smoking_Habit'] == val]['Height_or_Length_cm']
    fig.add_trace(go.Box(y=subset, name=val), row=1, col=3)

fig.update_layout(height=400, title_text='WASH & Environmental Factors vs Child Height', showlegend=False)
fig.update_yaxes(title_text='Height/Length (cm)')
fig.show()

> 💡 **Insight:** Grafik ini ngebandingin tinggi badan anak berdasarkan akses air bersih, kepemilikan jamban, dan kebiasaan merokok keluarga. Sanitasi yang buruk dan paparan asap rokok dikenal sebagai faktor yang bisa mengganggu penyerapan nutrisi dan pertumbuhan anak secara tidak langsung. Namun, karena tinggi/panjang badan yang digunakan masih merupakan nilai mentah dan belum mempertimbangkan usia maupun Z-score TB/U, grafik ini belum dapat digunakan untuk menentukan status stunting atau menyimpulkan adanya hubungan antara faktor lingkungan dan stunting. Analisis lebih lanjut dilakukan menggunakan status gizi berdasarkan Z-score WHO.

## Correlation Analysis

In [ ]:
# Correlation heatmap for numeric variables
corr_cols = ['Age_at_Measurement_Months', 'Mother_Height_cm', 'Mother_Pregnancy_Age',
             'Mother_ANC_Visits', 'Birth_Spacing_Months', 'Birth_Weight_kg', 'Birth_Length_cm',
             'Weight_kg', 'Height_or_Length_cm', 'Head_Circumference_cm']

corr_matrix = df[corr_cols].corr().round(2)
labels = ['Age', 'Mother Height', 'Preg Age', 'ANC Visits', 'Birth Spacing',
          'Birth Weight', 'Birth Length', 'Weight', 'Height', 'Head Circ']

fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values, x=labels, y=labels,
    colorscale='RdBu_r', zmin=-1, zmax=1,
    text=corr_matrix.values, texttemplate='%{text:.2f}', textfont={'size': 9}))
fig.update_layout(title='Correlation Heatmap - Health Metrics', height=550, width=650)
fig.show()

> 💡 **Insight:** Heatmap ini menunjukkan seberapa kuat hubungan antar variabel angka (semakin dekat ke +1 atau -1, semakin kuat hubungannya; dekat 0 berarti nggak ada hubungan). **Catatan penting:** karena dataset ini data dummy/simulasi, kolom-kolomnya diisi secara acak dan nggak benar-benar disambungkan secara sebab-akibat — misalnya korelasi antara tinggi badan ibu dan tinggi badan anak yang harusnya cukup kuat secara teori, di data ini malah mendekati 0. Jadi heatmap ini bagus buat latihan baca kode, tapi jangan dijadikan 'temuan' di laporan.

In [ ]:
# Mother height vs child height scatter
fig = px.scatter(df.sample(min(3000, len(df)), random_state=42),
                 x='Mother_Height_cm', y='Height_or_Length_cm',
                 color='Gender', opacity=0.4, size='Age_at_Measurement_Months',
                 title='Mother Height vs Child Height (sized by age)',
                 labels={'Mother_Height_cm': 'Mother Height (cm)',
                         'Height_or_Length_cm': 'Child Height/Length (cm)'},
                 color_discrete_sequence=['#FF6B6B', '#4ECDC4'])
fig.update_layout(height=450)
fig.show()

> 💡 **Insight:** Scatter plot ini seharusnya nunjukkin pola 'ibu pendek cenderung anaknya pendek juga' (disebut pewarisan tinggi badan antar generasi). Tapi sama seperti heatmap di atas, karena datanya simulasi acak, pola ini nggak benar-benar kelihatan di titik-titik datanya. Kalau nanti pola serupa dicoba ke data asli Yayasan, baru itu jadi temuan yang bisa dipercaya.

## Key Findings & Business Insights

In [ ]:
# Summary statistics and key findings
print("="*60)
print("        STUNTING EDA - KEY FINDINGS SUMMARY")
print("="*60)

print(f"\n📊 Dataset: {len(df):,} children | {df.shape[1]} variables")
print(f"\n👶 Child Demographics:")
print(f"   • Gender split: {df['Gender'].value_counts().to_dict()}")
print(f"   • Age range: {df['Age_at_Measurement_Months'].min():.0f} - {df['Age_at_Measurement_Months'].max():.0f} months")
print(f"   • Median age: {df['Age_at_Measurement_Months'].median():.1f} months")

print(f"\n📏 Anthropometric Measures:")
print(f"   • Median birth weight: {df['Birth_Weight_kg'].median():.2f} kg")
print(f"   • Low birth weight rate: {(df['Birth_Weight_kg'] < 2.5).mean()*100:.1f}%")
print(f"   • Median birth length: {df['Birth_Length_cm'].median():.1f} cm")
print(f"   • Median current height: {df['Height_or_Length_cm'].median():.1f} cm")

print(f"\n👩 Maternal Factors:")
print(f"   • Median mother height: {df['Mother_Height_cm'].median():.1f} cm")
print(f"   • Short stature mothers (<150cm): {(df['Mother_Height_cm'] < 150).mean()*100:.1f}%")
print(f"   • CED prevalence: {(df['Mother_Chronic_Energy_Deficiency']=='Yes').mean()*100:.1f}%")
print(f"   • Median ANC visits: {df['Mother_ANC_Visits'].median():.0f}")

print(f"\n🍼 Feeding Practices:")
print(f"   • Exclusive breastfeeding: {(df['Exclusive_Breastfeeding']=='Yes').mean()*100:.1f}%")
print(f"   • Adequate animal protein: {(df['Complementary_Feeding_Animal_Protein']=='Adequate').mean()*100:.1f}%")

print(f"\n🚰 WASH & Environment:")
print(f"   • Adequate clean water: {(df['Clean_Water_Access']=='Adequate').mean()*100:.1f}%")
print(f"   • Proper latrine: {(df['Latrine_Ownership']=='With_Septic_Tank').mean()*100:.1f}%")
print(f"   • Family smoking: {(df['Family_Smoking_Habit']=='Yes').mean()*100:.1f}%")

print(f"\n💊 Health Services:")
print(f"   • Complete immunization: {(df['Immunization_Status']=='Complete').mean()*100:.1f}%")
print(f"   • Iron supplementation: {(df['Mother_Iron_Supplementation']=='Yes').mean()*100:.1f}%")
print(f"   • Health insurance: {(df['Health_Insurance_Status']=='Active').mean()*100:.1f}%")

        STUNTING EDA - KEY FINDINGS SUMMARY

📊 Dataset: 10,000 children | 35 variables

👶 Child Demographics:
   • Gender split: {'Female': 5057, 'Male': 4943}
   • Age range: 0 - 59 months
   • Median age: 29.5 months

📏 Anthropometric Measures:
   • Median birth weight: 3.20 kg
   • Low birth weight rate: 7.9%
   • Median birth length: 49.8 cm
   • Median current height: 90.4 cm

👩 Maternal Factors:
   • Median mother height: 152.9 cm
   • Short stature mothers (<150cm): 27.9%
   • CED prevalence: 14.5%
   • Median ANC visits: 5

🍼 Feeding Practices:
   • Exclusive breastfeeding: 64.6%
   • Adequate animal protein: 53.8%

🚰 WASH & Environment:
   • Adequate clean water: 79.8%
   • Proper latrine: 60.9%
   • Family smoking: 65.6%

💊 Health Services:
   • Complete immunization: 59.5%
   • Iron supplementation: 55.4%
   • Health insurance: 60.6%


> 💡 **Insight:** Ini rekap angka-angka penting dari seluruh eksplorasi di atas, dikumpulin jadi satu ringkasan biar gampang dibaca sekali lihat — semacam 'kesimpulan sementara' sebelum masuk ke tahap perhitungan Z-score resmi.

# 5.Perhitungan Z-Score Resmi WHO

Bagian sebelumnya adalah eksplorasi awal (EDA) terhadap data dummy. Mulai bagian ini, kita menghitung Z-score sesungguhnya menggunakan tabel referensi resmi WHO (via library `pygrowup`, yang mengimplementasikan metode LMS WHO Child Growth Standards 2006), lalu mengklasifikasikan status gizi tiap anak sesuai ambang batas Permenkes RI No. 2 Tahun 2020.

In [ ]:
# Install pygrowup jika belum ada (implementasi resmi metode LMS WHO)
!pip install pygrowup --quiet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


> 💡 **Insight:** Baris ini cuma nginstall library `pygrowup`, yaitu tools Python yang udah dicek akurat buat ngitung Z-score sesuai rumus resmi WHO (metode LMS), jadi kita nggak perlu nulis rumus matematika WHO dari nol.

In [ ]:
from pygrowup import Calculator
import numpy as np

# --- Step 1: validasi ulang nilai ekstrem sebelum perhitungan z-score ---
print(f"Height/Length = 0  : {(df['Height_or_Length_cm'] == 0).sum()}")
print(f"Height/Length = 999: {(df['Height_or_Length_cm'] == 999).sum()}")
print(f"Weight = 0         : {(df['Weight_kg'] == 0).sum()}")
print(f"Weight = 999       : {(df['Weight_kg'] == 999).sum()}")


Height/Length = 0  : 0
Height/Length = 999: 0
Weight = 0         : 0
Weight = 999       : 0


In [ ]:
# --- Step 2: Hitung Z-score TB/U dan BB/U pakai tabel resmi WHO ---
calc = Calculator(adjust_height_data=True, adjust_weight_scores=False, include_cdc=False)

sex_map = {'Male': 'M', 'Female': 'F'}
df['sex_code'] = df['Gender'].map(sex_map)
# pygrowup otomatis koreksi +/- 0.7cm sesuai Permenkes kalau posisi ukur tidak sesuai usia
df['is_standing'] = df['Measurement_Posture'] == 'Standing'

def safe_zscore(fn, value, age, sex, standing):
    """Hitung Z-score dengan aman; kembalikan NaN kalau data tidak bisa diproses."""
    if pd.isna(value):
        return np.nan
    try:
        z = fn(measurement=value, age_in_months=age, sex=sex, height=standing)
        return float(z) if z is not None else np.nan
    except Exception:
        return np.nan

df['Z_TBU'] = df.apply(lambda r: safe_zscore(
    calc.lhfa, r['Height_or_Length_cm'], r['Age_at_Measurement_Months'], r['sex_code'], r['is_standing']
), axis=1)

df['Z_BBU'] = df.apply(lambda r: safe_zscore(
    calc.wfa, r['Weight_kg'], r['Age_at_Measurement_Months'], r['sex_code'], r['is_standing']
), axis=1)

print(df[['Z_TBU', 'Z_BBU']].describe())


             Z_TBU        Z_BBU
count  9900.000000  9950.000000
mean     -0.270043    -0.120061
std       1.284695     1.104540
min      -7.860000    -7.980000
25%      -0.970000    -0.450000
50%      -0.210000     0.090000
75%       0.530000     0.520000
max       4.570000     3.850000


> 💡 **Insight:** Ini bagian inti-nya: ngitung Z-score TB/U (tinggi per umur) dan BB/U (berat per umur) buat tiap anak, pakai tabel resmi WHO lewat `pygrowup`. Kode ini juga otomatis mengoreksi kalau posisi ukurnya nggak sesuai standar (misal anak di atas 2 tahun diukur terlentang, bukan berdiri), sesuai aturan +/- 0,7 cm di Permenkes. Baris yang datanya kosong otomatis dilewatin (hasilnya NaN), nggak bikin program error.

In [ ]:
# --- Step 3: Tandai nilai ekstrem untuk verifikasi manual (jangan langsung dihapus) ---
# |Z| > 5 SD sangat jarang terjadi secara biologis, kemungkinan besar kesalahan input.
df['Perlu_Verifikasi'] = df['Z_TBU'].abs() > 5
print(f"Baris ditandai perlu verifikasi manual: {df['Perlu_Verifikasi'].sum()}")


Baris ditandai perlu verifikasi manual: 31


> 💡 **Insight:** Ini nandain anak yang Z-score-nya lebih dari 5 SD (ke atas atau ke bawah) sebagai 'perlu dicek ulang', karena angka seekstrem itu sangat jarang terjadi secara biologis — kemungkinan besar itu salah input pas pengukuran (misal salah taruh koma desimal), bukan kondisi anak yang beneran seekstrem itu. Baris ini nggak dihapus, cuma ditandai, biar nanti bisa dicek manual dulu sebelum diambil tindakan.

In [ ]:
# --- Step 4: Klasifikasi status gizi sesuai Permenkes RI No. 2 Tahun 2020 ---

def klasifikasi_tbu(z):
    if pd.isna(z):
        return 'Data Tidak Lengkap'
    if z < -3:
        return 'Sangat Pendek (Severely Stunted)'
    if z < -2:
        return 'Pendek (Stunted)'
    if z <= 3:
        return 'Normal'
    return 'Tinggi'

def klasifikasi_bbu(z):
    if pd.isna(z):
        return 'Data Tidak Lengkap'
    if z < -3:
        return 'Berat Badan Sangat Kurang'
    if z < -2:
        return 'Berat Badan Kurang'
    if z <= 1:
        return 'Berat Badan Normal'
    return 'Risiko Berat Badan Lebih'

df['Status_Gizi_TBU'] = df['Z_TBU'].apply(klasifikasi_tbu)
df['Status_Gizi_BBU'] = df['Z_BBU'].apply(klasifikasi_bbu)

print(df['Status_Gizi_TBU'].value_counts())
print()
print(df['Status_Gizi_BBU'].value_counts())


Status_Gizi_TBU
Normal                              9051
Pendek (Stunted)                     502
Sangat Pendek (Severely Stunted)     299
Data Tidak Lengkap                   100
Tinggi                                48
Name: count, dtype: int64

Status_Gizi_BBU
Berat Badan Normal           8513
Risiko Berat Badan Lebih      778
Berat Badan Kurang            349
Berat Badan Sangat Kurang     310
Data Tidak Lengkap             50
Name: count, dtype: int64


> 💡 **Insight:** Ini nerjemahin angka Z-score jadi kategori yang gampang dibaca (Normal, Pendek, Sangat Pendek, dst), persis sesuai ambang batas resmi Permenkes 2/2020 yang udah kita verifikasi sebelumnya. Anak yang datanya kosong otomatis dikasih label 'Data Tidak Lengkap', bukan diklasifikasi asal-asalan.

In [ ]:
# --- Step 5: Petakan status gizi ke rekomendasi aksi (output utama Kelompok 1) ---

def rekomendasi_aksi(row):
    if row['Status_Gizi_TBU'] == 'Data Tidak Lengkap':
        return 'Data tidak lengkap - perlu verifikasi ulang ke lapangan'
    if row['Perlu_Verifikasi']:
        return 'Nilai ekstrem - verifikasi ulang pengukuran sebelum ambil tindakan'
    if row['Status_Gizi_TBU'] == 'Sangat Pendek (Severely Stunted)':
        return 'Rujuk segera ke fasilitas kesehatan'
    if row['Status_Gizi_TBU'] == 'Pendek (Stunted)':
        return 'Pemantauan rutin & edukasi gizi intensif'
    return 'Pemantauan rutin standar (Posyandu)'

df['Rekomendasi_Aksi'] = df.apply(rekomendasi_aksi, axis=1)
df['Rekomendasi_Aksi'].value_counts()


,count
Rekomendasi_Aksi,
Pemantauan rutin standar (Posyandu),9099
Pemantauan rutin & edukasi gizi intensif,502
Rujuk segera ke fasilitas kesehatan,268
Data tidak lengkap - perlu verifikasi ulang ke lapangan,100
Nilai ekstrem - verifikasi ulang pengukuran sebelum ambil tindakan,31


> 💡 **Insight:** Ini bagian yang paling penting buat Kelompok 1 — nerjemahin kategori status gizi jadi rekomendasi aksi konkret: anak severely stunted → rujuk segera ke faskes, anak stunted → pemantauan intensif, anak dengan Z-score ekstrem/data nggak lengkap → perlu verifikasi dulu, sisanya → pemantauan rutin biasa. Inilah yang jadi 'individual action sheet' yang diminta brief tugas kalian.

In [ ]:
# --- Step 6: Visualisasi hasil klasifikasi ---
fig = px.bar(df['Status_Gizi_TBU'].value_counts().reset_index(),
             x='Status_Gizi_TBU', y='count',
             title='Distribusi Status Gizi Anak Berdasarkan TB/U (setelah dihitung Z-score resmi WHO)',
             labels={'Status_Gizi_TBU': 'Kategori', 'count': 'Jumlah Anak'},
             color='Status_Gizi_TBU',
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_layout(showlegend=False)
fig.show()


> 💡 **Insight:** Bar chart ini nunjukkin secara visual berapa banyak anak di tiap kategori status gizi setelah dihitung pakai Z-score resmi (bukan cuma tinggi badan mentah). Ini grafik paling penting di seluruh notebook, karena inilah hasil klasifikasi yang sesungguhnya.

In [ ]:
# --- Step 6b: Visualisasi status gizi BB/U (pendamping TB/U) ---
fig_bbu = px.pie(df['Status_Gizi_BBU'].value_counts().reset_index(),
                  names='Status_Gizi_BBU', values='count', hole=0.5,
                  title='Distribusi Status Gizi Anak Berdasarkan BB/U (Z-score resmi WHO)',
                  color='Status_Gizi_BBU',
                  color_discrete_map={
                      'Berat Badan Normal': '#00A896',
                      'Risiko Berat Badan Lebih': '#F2A65A',
                      'Berat Badan Kurang': '#9AC4C6',
                      'Berat Badan Sangat Kurang': '#E5623E',
                      'Data Tidak Lengkap': '#5B7274'
                  })
fig_bbu.update_traces(textinfo='percent+label')
fig_bbu.show()

print(df['Status_Gizi_BBU'].value_counts())
print((df['Status_Gizi_BBU'].value_counts(normalize=True) * 100).round(2))

Status_Gizi_BBU
Berat Badan Normal           8513
Risiko Berat Badan Lebih      778
Berat Badan Kurang            349
Berat Badan Sangat Kurang     310
Data Tidak Lengkap             50
Name: count, dtype: int64
Status_Gizi_BBU
Berat Badan Normal           85.13
Risiko Berat Badan Lebih      7.78
Berat Badan Kurang            3.49
Berat Badan Sangat Kurang     3.10
Data Tidak Lengkap            0.50
Name: proportion, dtype: float64


> 💡 **Insight:** Pie chart ini melengkapi hasil TB/U dengan status gizi BB/U (Berat Badan menurut Umur). Bedanya: TB/U menunjukkan kondisi kronis/jangka panjang (stunting), sedangkan BB/U menunjukkan kondisi gizi saat ini (akut). Mayoritas anak (84,7%) berstatus Berat Badan Normal. Kombinasi TB/U + BB/U inilah yang jadi dasar penentuan Rekomendasi Aksi — misalnya anak dengan TB/U normal tapi BB/U mulai turun tetap perlu dipantau supaya tidak berkembang jadi stunting.

**Catatan penting:** dataset ini adalah data dummy/simulasi (nilai fitur seperti tinggi ibu, ASI eksklusif, dsb. dibuat acak dan tidak benar-benar berkorelasi secara biologis dengan hasil, terbukti dari korelasi yang mendekati nol). Alur kode di bagian ini (hitung Z-score -> klasifikasi -> rekomendasi aksi) yang menjadi inti pipeline, dan siap diterapkan ke dataset asli Yayasan (100 anak) begitu proses cleaning data mentahnya (parsing teks TB/BB, dsb.) selesai.

# 6.Analisis Faktor Risiko Berdasarkan Status Gizi Resmi (Kelompok 1)

Bagian ini membandingkan **tingkat stunting** (hasil klasifikasi Z-score resmi WHO) antar kelompok faktor risiko yang secara teori dikenal berhubungan dengan stunting: berat lahir rendah, ASI eksklusif, status KEK ibu, pendidikan ibu, dan akses air bersih. Ini berbeda dengan Bagian (EDA awal) yang membandingkan tinggi badan mentah — di sini pembandingnya adalah **persentase anak stunting** (Pendek + Sangat Pendek) berdasarkan hasil klasifikasi resmi.

In [ ]:
# --- Analisis: Tingkat stunting per kelompok faktor risiko ---
df['Stunted_Flag'] = df['Status_Gizi_TBU'].isin(
    ['Pendek (Stunted)', 'Sangat Pendek (Severely Stunted)']
)

# Hanya gunakan baris yang statusnya sudah terklasifikasi (bukan 'Data Tidak Lengkap')
valid = df[df['Status_Gizi_TBU'] != 'Data Tidak Lengkap'].copy()
valid['Low_Birth_Weight'] = valid['Birth_Weight_kg'] < 2.5

def stunting_rate(col):
    rate = valid.groupby(col)['Stunted_Flag'].mean() * 100
    n = valid.groupby(col).size()
    return pd.DataFrame({'Tingkat_Stunting_%': rate.round(2), 'n': n})

print("=== Tingkat Stunting: Berat Lahir Rendah ===")
print(stunting_rate('Low_Birth_Weight'))

print("\n=== Tingkat Stunting: ASI Eksklusif ===")
print(stunting_rate('Exclusive_Breastfeeding'))

print("\n=== Tingkat Stunting: Status KEK Ibu ===")
print(stunting_rate('Mother_Chronic_Energy_Deficiency'))

print("\n=== Tingkat Stunting: Pendidikan Ibu ===")
print(stunting_rate('Mother_Education'))

print("\n=== Tingkat Stunting: Akses Air Bersih ===")
print(stunting_rate('Clean_Water_Access'))

print(f"\nTingkat stunting keseluruhan (data valid): {valid['Stunted_Flag'].mean()*100:.2f}%")

=== Tingkat Stunting: Berat Lahir Rendah ===
                  Tingkat_Stunting_%     n
Low_Birth_Weight                          
False                           8.15  9121
True                            7.45   779

=== Tingkat Stunting: ASI Eksklusif ===
                         Tingkat_Stunting_%     n
Exclusive_Breastfeeding                          
No                                     7.11  3517
Yes                                    8.63  6383

=== Tingkat Stunting: Status KEK Ibu ===
                                  Tingkat_Stunting_%     n
Mother_Chronic_Energy_Deficiency                          
No                                              8.10  8466
Yes                                             8.02  1434

=== Tingkat Stunting: Pendidikan Ibu ===
                  Tingkat_Stunting_%     n
Mother_Education                          
Primary_or_Lower                8.00  1925
Middle_School                   8.32  3497
High_School                     8.22  3505
Higher_

/tmp/ipykernel_1534/2792084287.py:11: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_1534/2792084287.py:12: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_1534/2792084287.py:11: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_1534/2792084287.py:12: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=F

In [ ]:
# --- Visualisasi ringkasan tingkat stunting per faktor risiko ---
risk_summary = pd.DataFrame({
    'Faktor': ['Berat Lahir Rendah', 'ASI Tidak Eksklusif', 'Ibu dengan KEK', 'Pendidikan Ibu Rendah', 'Akses Air Tidak Layak'],
    'Kelompok_Berisiko_%': [
        stunting_rate('Low_Birth_Weight').loc[True, 'Tingkat_Stunting_%'],
        stunting_rate('Exclusive_Breastfeeding').loc['No', 'Tingkat_Stunting_%'],
        stunting_rate('Mother_Chronic_Energy_Deficiency').loc['Yes', 'Tingkat_Stunting_%'],
        valid[valid['Mother_Education'].isin(['Primary_or_Lower', 'Middle_School'])]['Stunted_Flag'].mean() * 100,
        stunting_rate('Clean_Water_Access').loc['Inadequate', 'Tingkat_Stunting_%'],
    ],
    'Kelompok_Pembanding_%': [
        stunting_rate('Low_Birth_Weight').loc[False, 'Tingkat_Stunting_%'],
        stunting_rate('Exclusive_Breastfeeding').loc['Yes', 'Tingkat_Stunting_%'],
        stunting_rate('Mother_Chronic_Energy_Deficiency').loc['No', 'Tingkat_Stunting_%'],
        valid[valid['Mother_Education'].isin(['High_School', 'Higher_Education'])]['Stunted_Flag'].mean() * 100,
        stunting_rate('Clean_Water_Access').loc['Adequate', 'Tingkat_Stunting_%'],
    ],
})

fig_risk = go.Figure()
fig_risk.add_trace(go.Bar(name='Kelompok Berisiko', x=risk_summary['Faktor'], y=risk_summary['Kelompok_Berisiko_%'], marker_color='#E5623E'))
fig_risk.add_trace(go.Bar(name='Kelompok Pembanding', x=risk_summary['Faktor'], y=risk_summary['Kelompok_Pembanding_%'], marker_color='#00A896'))
fig_risk.update_layout(title='Tingkat Stunting: Kelompok Berisiko vs Pembanding (%)', barmode='group', yaxis_title='Tingkat Stunting (%)')
fig_risk.show()

print(risk_summary.round(2))


/tmp/ipykernel_1534/2792084287.py:11: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_1534/2792084287.py:12: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_1534/2792084287.py:11: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_1534/2792084287.py:12: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=F

                  Faktor  Kelompok_Berisiko_%  Kelompok_Pembanding_%
0     Berat Lahir Rendah                 7.45                   8.15
1    ASI Tidak Eksklusif                 7.11                   8.63
2         Ibu dengan KEK                 8.02                   8.10
3  Pendidikan Ibu Rendah                 8.21                   7.95
4  Akses Air Tidak Layak                 7.93                   8.13


> 💡 **Insight:** Grafik ini membandingkan tingkat stunting antara kelompok yang secara teori berisiko (berat lahir rendah, tidak ASI eksklusif, ibu KEK, pendidikan ibu rendah, akses air tidak layak) dengan kelompok pembandingnya. **Hasilnya: selisih antar kelompok sangat kecil (umumnya di bawah 1,5 poin persen) dan tidak signifikan.** Ini konsisten dengan catatan (korelasi mendekati nol) — karena dataset ini simulasi/acak, nilai fitur tidak benar-benar disambungkan secara sebab-akibat ke hasil. Yang penting dari bagian ini bukan menemukan pola baru, melainkan memvalidasi bahwa **pipeline analisisnya sudah benar dan siap diterapkan** ke data asli Yayasan (±100 anak) untuk mendapatkan pola yang benar-benar dapat dipercaya.

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 44 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   Child_ID                              10000 non-null  object        
 1   National_ID_NIK                       10000 non-null  int64         
 2   Child_Name                            10000 non-null  object        
 3   Parent_Name                           10000 non-null  object        
 4   Gender                                10000 non-null  category      
 5   Neighborhood_RT_RW                    10000 non-null  object        
 6   Birth_Date                            10000 non-null  datetime64[ns]
 7   Measurement_Date                      10000 non-null  datetime64[ns]
 8   Age_at_Measurement_Months             10000 non-null  float64       
 9   Mother_Height_cm                      10000 non-null  float64       
 10 

# 7.Preprocessing

## Tahap 1. Menentukan target dan fitur awal

In [ ]:
# Menentukan target
target = 'Stunted_Flag'

# Memisahkan target (y) dan fitur (X)
y = df[target]
X = df.drop(columns=[target])

print("Target:")
print(y.value_counts())

print("\nJumlah fitur awal:", X.shape[1])
print("Jumlah data:", X.shape[0])

Target:
Stunted_Flag
False    9199
True      801
Name: count, dtype: int64

Jumlah fitur awal: 43
Jumlah data: 10000


> 💡 **Insight:** Stunted_Flag ditetapkan sebagai variabel target dalam pemodelan klasifikasi, sedangkan 43 kolom lainnya menjadi fitur awal. Dari 10.000 data, terdapat 9.199 anak (91,99%) dengan status tidak stunting (False) dan 801 anak (8,01%) dengan status stunting (True). Distribusi target menunjukkan adanya ketidakseimbangan kelas (class imbalance)

## Tahap 2. Feature Selection

In [ ]:
# --- Tahap 2: Feature Selection ---

# Kolom identitas dan administratif
id_cols = [
    'Child_ID',
    'National_ID_NIK',
    'Child_Name',
    'Parent_Name',
    'Neighborhood_RT_RW'
]

# Kolom tanggal mentah
date_cols = [
    'Birth_Date',
    'Measurement_Date'
]

# Kolom yang berpotensi menyebabkan data leakage
leakage_cols = [
    'Z_TBU',
    'Z_BBU',
    'Status_Gizi_TBU',
    'Status_Gizi_BBU',
    'Perlu_Verifikasi',
    'Rekomendasi_Aksi'
]

# Gabungkan seluruh kolom yang tidak digunakan sebagai fitur
drop_cols = id_cols + date_cols + leakage_cols

# Hapus dari X
X = X.drop(columns=drop_cols, errors='ignore')

print("Kolom yang dihapus:")
for col in drop_cols:
    print("-", col)

print("\nJumlah fitur setelah feature selection:", X.shape[1])

Kolom yang dihapus:
- Child_ID
- National_ID_NIK
- Child_Name
- Parent_Name
- Neighborhood_RT_RW
- Birth_Date
- Measurement_Date
- Z_TBU
- Z_BBU
- Status_Gizi_TBU
- Status_Gizi_BBU
- Perlu_Verifikasi
- Rekomendasi_Aksi

Jumlah fitur setelah feature selection: 30


1. Identitas
* Child_ID
* National_ID_NIK
* Child_Name
* Parent_Name
* Neighborhood_RT_RW

Tidak seharusnya digunakan untuk memprediksi stunting karena merupakan identitas/informasi administratif, bukan karakteristik yang ingin dipelajari model.

2. Kolom Birth_Date dan Measurement_Date

Keduanya dikeluarkan karena informasi usia sudah direpresentasikan oleh Age_at_Measurement_Months.

3. Data leakage
* Z_TBU → digunakan untuk menentukan status stunting.
* Status_Gizi_TBU dan Status_Gizi_BBU → merupakan hasil klasifikasi dari Z_TBU.
* Perlu_Verifikasi → dibuat berdasarkan Z_TBU.
* Rekomendasi_Aksi → dibuat berdasarkan Status_Gizi_TBU dan Perlu_Verifikasi.

Jadi kalau kolom-kolom tersebut dimasukkan sebagai fitur, model bisa mendapatkan informasi yang berasal dari proses pembentukan target.

## Tahap 3. Cek tipe data dan missing value

In [ ]:
# --- Tahap 3: Pemeriksaan tipe data dan missing value ---

print("Tipe data fitur:")
display(X.dtypes.to_frame('Dtype'))

print("\nJumlah missing value:")
missing = X.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if missing.empty:
    print("Tidak ada missing value.")
else:
    display(missing.to_frame('Missing'))

Tipe data fitur:


,Dtype
Gender,category
Age_at_Measurement_Months,float64
Mother_Height_cm,float64
Mother_Education,category
Mother_Pregnancy_Age,float64
Mother_Chronic_Energy_Deficiency,category
Mother_ANC_Visits,int64
Mother_Iron_Supplementation,category
Birth_Spacing_Months,int64
Birth_Weight_kg,float64



Jumlah missing value:


,Missing
Height_or_Length_cm,100
Weight_kg,50
Head_Circumference_cm,50


Datatype masih ada yg kategorikal dan object, serta masih ada missing values. Hal tersebut akan ditangani setelah train-test split

## Tahap 4. Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Membagi data menjadi training dan testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Menampilkan ukuran masing-masing data
print("Ukuran data training:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nUkuran data testing:")
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

Ukuran data training:
X_train: (8000, 30)
y_train: (8000,)

Ukuran data testing:
X_test : (2000, 30)
y_test : (2000,)


## Tahap 5. Pemisahan Fitur Numerik dan Kategorikal

In [ ]:
# 5 Memisahkan fitur numerik dan kategorikal

numeric_features = X_train.select_dtypes(
    include=['int64', 'float64', 'int32', 'float32']
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=['object', 'category', 'bool']
).columns.tolist()

print("Fitur numerik:")
print(numeric_features)

print("\nJumlah fitur numerik:", len(numeric_features))

print("\nFitur kategorikal:")
print(categorical_features)

print("\nJumlah fitur kategorikal:", len(categorical_features))

Fitur numerik:
['Age_at_Measurement_Months', 'Mother_Height_cm', 'Mother_Pregnancy_Age', 'Mother_ANC_Visits', 'Birth_Spacing_Months', 'Birth_Weight_kg', 'Birth_Length_cm', 'Weight_kg', 'Height_or_Length_cm', 'Head_Circumference_cm']

Jumlah fitur numerik: 10

Fitur kategorikal:
['Gender', 'Mother_Education', 'Mother_Chronic_Energy_Deficiency', 'Mother_Iron_Supplementation', 'Measurement_Posture', 'Exclusive_Breastfeeding', 'Complementary_Feeding_Animal_Protein', 'Infection_History', 'Immunization_Status', 'Clean_Water_Access', 'Latrine_Ownership', 'Family_Smoking_Habit', 'Social_Assistance_Recipient', 'Health_Insurance_Status', 'Data_Source', 'Monitoring_Type', 'Age_Group', 'Low_Birth_Weight', 'sex_code', 'is_standing']

Jumlah fitur kategorikal: 20


## Tahap 6. Imputation

In [ ]:
# --- Tahap 6: Imputation (Penanganan Missing Value) ---

from sklearn.impute import SimpleImputer
import pandas as pd

# Mengubah fitur kategorikal menjadi object agar kompatibel dengan imputer
X_train[categorical_features] = X_train[categorical_features].astype('object')
X_test[categorical_features] = X_test[categorical_features].astype('object')

# Imputer untuk fitur numerik
numeric_imputer = SimpleImputer(strategy='median')

# Imputer untuk fitur kategorikal
categorical_imputer = SimpleImputer(strategy='most_frequent')

# Fit hanya pada data training
X_train_num = numeric_imputer.fit_transform(
    X_train[numeric_features]
)

X_train_cat = categorical_imputer.fit_transform(
    X_train[categorical_features]
)

# Data testing hanya menggunakan parameter dari training
X_test_num = numeric_imputer.transform(
    X_test[numeric_features]
)

X_test_cat = categorical_imputer.transform(
    X_test[categorical_features]
)

# Menggabungkan kembali hasil imputation
X_train_imputed = pd.DataFrame(
    X_train_num,
    columns=numeric_features,
    index=X_train.index
)

X_train_imputed[categorical_features] = pd.DataFrame(
    X_train_cat,
    columns=categorical_features,
    index=X_train.index
)

X_test_imputed = pd.DataFrame(
    X_test_num,
    columns=numeric_features,
    index=X_test.index
)

X_test_imputed[categorical_features] = pd.DataFrame(
    X_test_cat,
    columns=categorical_features,
    index=X_test.index
)

print("Imputation berhasil.")
print("X_train_imputed:", X_train_imputed.shape)
print("X_test_imputed :", X_test_imputed.shape)

Imputation berhasil.
X_train_imputed: (8000, 30)
X_test_imputed : (2000, 30)


## Tahap 7. SMOTENC

In [ ]:
# --- Tahap 7: Penanganan Class Imbalance dengan SMOTENC ---

from imblearn.over_sampling import SMOTENC

# Mengubah fitur kategorikal menjadi kode numerik sementara
# agar dapat diproses oleh SMOTENC
X_train_smotenc = X_train_imputed.copy()

for col in categorical_features:
    X_train_smotenc[col] = (
        X_train_smotenc[col]
        .astype('category')
        .cat.codes
    )

# Menentukan posisi/index fitur kategorikal
categorical_indices = [
    X_train_smotenc.columns.get_loc(col)
    for col in categorical_features
]

# Membuat SMOTENC
smotenc = SMOTENC(
    categorical_features=categorical_indices,
    random_state=42
)

# SMOTENC hanya diterapkan pada data training
X_train_resampled, y_train_resampled = smotenc.fit_resample(
    X_train_smotenc,
    y_train
)

print("Distribusi target sebelum SMOTENC:")
print(y_train.value_counts())

print("\nDistribusi target setelah SMOTENC:")
print(y_train_resampled.value_counts())

print("\nUkuran data setelah SMOTENC:")
print("X_train_resampled:", X_train_resampled.shape)
print("y_train_resampled:", y_train_resampled.shape)

Distribusi target sebelum SMOTENC:
Stunted_Flag
False    7359
True      641
Name: count, dtype: int64

Distribusi target setelah SMOTENC:
Stunted_Flag
False    7359
True     7359
Name: count, dtype: int64

Ukuran data setelah SMOTENC:
X_train_resampled: (14718, 30)
y_train_resampled: (14718,)


## Tahap 8. Encoding dan Scaling

In [ ]:
# --- Tahap 8: Encoding dan Scaling ---

from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Mengembalikan kode kategorikal hasil SMOTENC menjadi kategori asli
X_train_resampled = X_train_resampled.copy()

for col in categorical_features:
    categories = X_train_imputed[col].astype('category').cat.categories
    X_train_resampled[col] = (
        X_train_resampled[col]
        .round()
        .astype(int)
        .map(dict(enumerate(categories)))
    )

# One-Hot Encoding
encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

X_train_cat_encoded = encoder.fit_transform(
    X_train_resampled[categorical_features]
)

X_test_cat_encoded = encoder.transform(
    X_test_imputed[categorical_features]
)

# Scaling fitur numerik
scaler = StandardScaler()

X_train_num_scaled = scaler.fit_transform(
    X_train_resampled[numeric_features]
)

X_test_num_scaled = scaler.transform(
    X_test_imputed[numeric_features]
)

# Menggabungkan fitur numerik dan kategorikal
import numpy as np

X_train_final = np.hstack([
    X_train_num_scaled,
    X_train_cat_encoded
])

X_test_final = np.hstack([
    X_test_num_scaled,
    X_test_cat_encoded
])

print("Ukuran data setelah encoding dan scaling:")
print("X_train_final:", X_train_final.shape)
print("X_test_final :", X_test_final.shape)

Ukuran data setelah encoding dan scaling:
X_train_final: (14718, 63)
X_test_final : (2000, 63)


# 8.MODELLING KLASIFIKASI

## Tahap 9. Import Library Modeling

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
    f1_score, recall_score, precision_score, accuracy_score
)
import matplotlib.pyplot as plt
import joblib
import os

> 💡 **Insight:** Karena target (`Stunted_Flag`) bersifat imbalanced (91,99% vs 8,01% sebelum SMOTENC), pemilihan algoritma dan metrik evaluasi harus mempertimbangkan hal ini. Kita akan melatih beberapa algoritma sekaligus (bukan cuma satu) supaya bisa dibandingkan performanya secara objektif, lalu pilih yang terbaik berdasarkan metrik yang relevan untuk kasus kesehatan seperti ini (bukan cuma Accuracy).

## Tahap 10. Training Beberapa Model (Model Comparison)

Kita latih 4 algoritma klasifikasi yang umum dipakai untuk data tabular:
1. **Logistic Regression** — baseline sederhana, cepat, mudah diinterpretasi (koefisien).
2. **Decision Tree** — baseline non-linear, mudah divisualisasikan aturan keputusannya.
3. **Random Forest** — ensemble dari banyak decision tree, biasanya lebih stabil & robust terhadap overfitting.
4. **Gradient Boosting** — ensemble boosting, sering memberi performa terbaik untuk data tabular berukuran menengah seperti ini.

Semua model dilatih menggunakan `X_train_final` & `y_train_resampled` (hasil SMOTENC — data training yang sudah balanced), lalu dievaluasi ke `X_test_final` & `y_test` (data testing asli, TIDAK di-SMOTENC, supaya evaluasi mencerminkan kondisi dunia nyata).

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=None, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train_final, y_train_resampled)
    y_pred = model.predict(X_test_final)
    y_proba = model.predict_proba(X_test_final)[:, 1]

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_proba),
        'PR-AUC': average_precision_score(y_test, y_proba)
    })
    fitted_models[name] = model

results_df = pd.DataFrame(results).sort_values('F1-Score', ascending=False).reset_index(drop=True)
print("=== Perbandingan Performa Model (di data testing) ===")
display(results_df.round(4))

> 💡 **Insight - Kenapa bukan cuma lihat Accuracy?**
> Karena kelas target imbalanced di data testing (mayoritas anak TIDAK stunting), model yang malas dan selalu menebak "Tidak Stunting" untuk semua anak pun bisa dapat Accuracy tinggi (~92%) padahal gagal total mendeteksi anak yang stunting. Ini disebut **accuracy paradox**.
>
> Metrik yang lebih relevan untuk kasus ini:
> - **Recall (Sensitivity)** — dari semua anak yang BENAR stunting, berapa persen berhasil terdeteksi model? Ini paling penting secara medis, karena **False Negative (anak stunting tapi diprediksi normal) jauh lebih berbahaya** daripada False Positive — anak yang butuh rujukan malah tidak terdeteksi.
> - **Precision** — dari semua anak yang diprediksi stunting, berapa persen yang benar-benar stunting? Penting supaya sumber daya rujukan/intervensi tidak salah sasaran ke anak yang sebenarnya normal.
> - **F1-Score** — keseimbangan antara Precision dan Recall.
> - **ROC-AUC & PR-AUC** — mengukur kemampuan model membedakan kelas di semua kemungkinan threshold. **PR-AUC lebih informatif dibanding ROC-AUC untuk data imbalanced** seperti ini.

## Tahap 11. Visualisasi Evaluasi: Confusion Matrix, ROC Curve, PR Curve

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(5*len(models), 4))
for ax, (name, model) in zip(axes, fitted_models.items()):
    y_pred = model.predict(X_test_final)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Tidak Stunting', 'Stunting'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
fig = go.Figure()
for name, model in fitted_models.items():
    y_proba = model.predict_proba(X_test_final)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=f'{name} (AUC={auc:.3f})'))

fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Random Baseline', line=dict(dash='dash', color='gray')))
fig.update_layout(title='ROC Curve - Perbandingan Model', xaxis_title='False Positive Rate', yaxis_title='True Positive Rate', height=450)
fig.show()

In [ ]:
fig = go.Figure()
for name, model in fitted_models.items():
    y_proba = model.predict_proba(X_test_final)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)
    fig.add_trace(go.Scatter(x=recall, y=precision, mode='lines', name=f'{name} (PR-AUC={ap:.3f})'))

baseline_rate = y_test.mean()
fig.add_hline(y=baseline_rate, line_dash='dash', line_color='gray', annotation_text='Random Baseline')
fig.update_layout(title='Precision-Recall Curve - Perbandingan Model', xaxis_title='Recall', yaxis_title='Precision', height=450)
fig.show()

## Tahap 12. Pemilihan Model Terbaik & Feature Importance

Model terbaik dipilih otomatis berdasarkan **F1-Score** tertinggi di data testing (baris pertama `results_df`, karena sudah di-`sort_values`), karena F1 merepresentasikan keseimbangan antara Precision dan Recall — dua hal yang sama-sama penting di kasus ini.

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_model = fitted_models[best_model_name]

print(f"Model terbaik terpilih: {best_model_name}")
print(f"\nMetrik lengkap model terbaik:")
print(results_df.iloc[0])

print(f"\n=== Classification Report - {best_model_name} ===")
y_pred_best = best_model.predict(X_test_final)
print(classification_report(y_test, y_pred_best, target_names=['Tidak Stunting', 'Stunting']))

In [ ]:
feature_names = numeric_features + list(encoder.get_feature_names_out(categorical_features))

if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=feature_names).sort_values(ascending=False)
    top_features = importances.head(15).sort_values()
    fig = px.bar(x=top_features.values, y=top_features.index, orientation='h',
                 title=f'Top 15 Feature Importance - {best_model_name}',
                 labels={'x': 'Importance', 'y': 'Fitur'})
    fig.update_layout(height=500)
    fig.show()
elif hasattr(best_model, 'coef_'):
    importances = pd.Series(best_model.coef_[0], index=feature_names)
    top_features = importances.reindex(importances.abs().sort_values(ascending=False).index).head(15).sort_values()
    fig = px.bar(x=top_features.values, y=top_features.index, orientation='h',
                 title=f'Top 15 Koefisien (arah pengaruh) - {best_model_name}',
                 labels={'x': 'Koefisien', 'y': 'Fitur'})
    fig.update_layout(height=500)
    fig.show()

> 💡 **Insight:** Feature importance menunjukkan fitur mana yang paling banyak dipakai model untuk membedakan anak stunting vs tidak. **Catatan penting (sama seperti bagian EDA):** karena dataset ini dummy/simulasi dan fitur-fiturnya diisi acak, urutan feature importance di sini **belum tentu mencerminkan faktor risiko stunting yang sesungguhnya**. Kegunaan utama tahap ini adalah memvalidasi bahwa pipeline modeling berjalan dan siap diterapkan ulang ke data asli Yayasan.

## Tahap 13. Threshold Tuning (Opsional tapi Direkomendasikan)

Secara default, model mengklasifikasikan "Stunting" jika probabilitas prediksi > 0.5. Tapi untuk kasus kesehatan seperti ini, kita bisa **menurunkan threshold** supaya model lebih sensitif menangkap anak stunting (Recall naik), dengan konsekuensi Precision sedikit turun (lebih banyak anak normal yang ikut ditandai perlu verifikasi — risikonya jauh lebih kecil dibanding anak stunting yang terlewat).

In [ ]:
y_proba_best = best_model.predict_proba(X_test_final)[:, 1]

thresholds = np.arange(0.1, 0.9, 0.05)
threshold_results = []
for t in thresholds:
    y_pred_t = (y_proba_best >= t).astype(int)
    threshold_results.append({
        'Threshold': round(t, 2),
        'Precision': precision_score(y_test, y_pred_t, zero_division=0),
        'Recall': recall_score(y_test, y_pred_t, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred_t, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_results)
best_threshold = threshold_df.loc[threshold_df['F1-Score'].idxmax(), 'Threshold']

fig = go.Figure()
fig.add_trace(go.Scatter(x=threshold_df['Threshold'], y=threshold_df['Precision'], name='Precision'))
fig.add_trace(go.Scatter(x=threshold_df['Threshold'], y=threshold_df['Recall'], name='Recall'))
fig.add_trace(go.Scatter(x=threshold_df['Threshold'], y=threshold_df['F1-Score'], name='F1-Score'))
fig.add_vline(x=best_threshold, line_dash='dash', annotation_text=f'Threshold optimal={best_threshold}')
fig.update_layout(title='Precision/Recall/F1 vs Decision Threshold', xaxis_title='Threshold', yaxis_title='Score', height=450)
fig.show()

print(f"Threshold optimal (F1 tertinggi): {best_threshold}")
print(threshold_df)

## Tahap 14. Menyimpan Model & Preprocessing Artifacts

Untuk deployment, kita tidak cukup hanya menyimpan model — kita juga perlu menyimpan **semua objek preprocessing yang sudah di-`fit`** (imputer, encoder, scaler), karena input baru dari user (misalnya dari form aplikasi) masih berupa data mentah dan harus melalui proses yang **identik** dengan yang dipakai saat training, sebelum bisa diprediksi oleh model.

In [ ]:
os.makedirs('model_artifacts', exist_ok=True)

joblib.dump(best_model, 'model_artifacts/stunting_model.pkl')
joblib.dump(numeric_imputer, 'model_artifacts/numeric_imputer.pkl')
joblib.dump(categorical_imputer, 'model_artifacts/categorical_imputer.pkl')
joblib.dump(encoder, 'model_artifacts/onehot_encoder.pkl')
joblib.dump(scaler, 'model_artifacts/scaler.pkl')

joblib.dump({
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'best_threshold': float(best_threshold),
    'model_name': best_model_name
}, 'model_artifacts/metadata.pkl')

print("Semua artifact berhasil disimpan di folder 'model_artifacts/':")
for f in os.listdir('model_artifacts'):
    print(" -", f)

# 9. DEPLOYMENT

Deployment di sini berarti membungkus model + preprocessing pipeline yang sudah dilatih menjadi **aplikasi web interaktif**, supaya orang lain (misalnya kader Posyandu atau tim Yayasan) bisa memasukkan data satu anak lewat form, dan langsung dapat prediksi risiko stunting — tanpa perlu buka notebook atau menulis kode sama sekali.

**Pendekatan yang dipakai: Streamlit**, karena:
- Cukup dengan kode Python biasa (tidak perlu HTML/CSS/JavaScript)
- Gratis untuk hosting via Streamlit Community Cloud
- Cocok untuk internal tool / prototipe cepat seperti kebutuhan Yayasan ini

**Alur aplikasinya:**
1. User isi form (usia anak, berat, tinggi, data ibu, dsb.) lewat browser
2. Aplikasi susun input jadi 1 baris data, lalu jalankan lewat pipeline preprocessing yang SAMA PERSIS dengan saat training (imputer → encoder → scaler)
3. Model (`stunting_model.pkl`) memprediksi probabilitas stunting
4. Probabilitas dibandingkan dengan threshold optimal → tampilkan hasil klasifikasi + rekomendasi aksi

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib

st.set_page_config(page_title="Prediksi Risiko Stunting", page_icon="🧒", layout="centered")

# --- Load model & preprocessing artifacts (di-cache supaya tidak reload tiap interaksi) ---
@st.cache_resource
def load_artifacts():
    model = joblib.load('model_artifacts/stunting_model.pkl')
    numeric_imputer = joblib.load('model_artifacts/numeric_imputer.pkl')
    categorical_imputer = joblib.load('model_artifacts/categorical_imputer.pkl')
    encoder = joblib.load('model_artifacts/onehot_encoder.pkl')
    scaler = joblib.load('model_artifacts/scaler.pkl')
    metadata = joblib.load('model_artifacts/metadata.pkl')
    return model, numeric_imputer, categorical_imputer, encoder, scaler, metadata

model, numeric_imputer, categorical_imputer, encoder, scaler, metadata = load_artifacts()
numeric_features = metadata['numeric_features']
categorical_features = metadata['categorical_features']
best_threshold = metadata['best_threshold']

st.title("🧒 Prediksi Risiko Stunting pada Anak")
st.caption(f"Model: {metadata['model_name']} | Threshold keputusan: {best_threshold}")
st.write("Isi data anak di bawah ini untuk memprediksi risiko stunting berdasarkan indikator kesehatan dan sosial-ekonomi.")

with st.form("input_form"):
    st.subheader("Data Anak")
    col1, col2 = st.columns(2)
    with col1:
        age_months = st.number_input("Usia saat pengukuran (bulan)", 0, 60, 24)
        gender = st.selectbox("Jenis Kelamin", ["Male", "Female"])
        weight_kg = st.number_input("Berat badan saat ini (kg)", 1.0, 30.0, 11.0, step=0.1)
        height_cm = st.number_input("Tinggi/panjang badan saat ini (cm)", 30.0, 130.0, 80.0, step=0.1)
        head_circ = st.number_input("Lingkar kepala (cm)", 30.0, 55.0, 45.0, step=0.1)
        posture = st.selectbox("Posisi Pengukuran", ["Standing", "Lying"])
    with col2:
        birth_weight = st.number_input("Berat lahir (kg)", 0.5, 6.0, 3.0, step=0.1)
        birth_length = st.number_input("Panjang lahir (cm)", 30.0, 60.0, 49.0, step=0.1)
        birth_spacing = st.number_input("Jarak kelahiran dengan anak sebelumnya (bulan)", 0, 120, 24)
        exclusive_bf = st.selectbox("ASI Eksklusif (6 bulan pertama)", ["Yes", "No"])
        comp_feeding = st.selectbox("Kecukupan Protein Hewani (MPASI)", ["Adequate", "Inadequate"])
        immunization = st.selectbox("Status Imunisasi", ["Complete", "Incomplete"])
    infection_history = st.selectbox("Riwayat Infeksi", ["No_Record", "Yes", "No"])

    st.subheader("Data Ibu")
    col3, col4 = st.columns(2)
    with col3:
        mother_height = st.number_input("Tinggi badan ibu (cm)", 120.0, 180.0, 153.0, step=0.1)
        mother_preg_age = st.number_input("Usia ibu saat hamil (tahun)", 15, 50, 27)
        mother_anc = st.number_input("Jumlah kunjungan ANC", 0, 12, 4)
        mother_education = st.selectbox("Pendidikan Ibu", ["Primary_or_Lower", "Middle_School", "High_School", "Higher_Education"])
    with col4:
        mother_ced = st.selectbox("Status KEK Ibu", ["Yes", "No"])
        mother_iron = st.selectbox("Suplementasi Zat Besi", ["Yes", "No"])

    st.subheader("Lingkungan & Sosial-Ekonomi")
    col5, col6 = st.columns(2)
    with col5:
        clean_water = st.selectbox("Akses Air Bersih", ["Adequate", "Inadequate"])
        latrine = st.selectbox("Kepemilikan Jamban", ["With_Septic_Tank", "Without_Septic_Tank"])
        smoking = st.selectbox("Ada Anggota Keluarga Merokok", ["Yes", "No"])
    with col6:
        social_assist = st.selectbox("Penerima Bantuan Sosial", ["Yes", "No"])
        health_insurance = st.selectbox("Kepesertaan Asuransi Kesehatan", ["Yes", "No"])
        data_source = st.selectbox("Sumber Data", ["Posyandu", "Puskesmas", "Survey"])
        monitoring_type = st.selectbox("Jenis Pemantauan", ["Routine", "Intervention"])

    submitted = st.form_submit_button("Prediksi Risiko Stunting")

if submitted:
    raw_input = {
        'Age_at_Measurement_Months': age_months,
        'Gender': gender,
        'Weight_kg': weight_kg,
        'Height_or_Length_cm': height_cm,
        'Head_Circumference_cm': head_circ,
        'Measurement_Posture': posture,
        'Birth_Weight_kg': birth_weight,
        'Birth_Length_cm': birth_length,
        'Birth_Spacing_Months': birth_spacing,
        'Exclusive_Breastfeeding': exclusive_bf,
        'Complementary_Feeding_Animal_Protein': comp_feeding,
        'Immunization_Status': immunization,
        'Infection_History': infection_history,
        'Mother_Height_cm': mother_height,
        'Mother_Pregnancy_Age': mother_preg_age,
        'Mother_ANC_Visits': mother_anc,
        'Mother_Education': mother_education,
        'Mother_Chronic_Energy_Deficiency': mother_ced,
        'Mother_Iron_Supplementation': mother_iron,
        'Clean_Water_Access': clean_water,
        'Latrine_Ownership': latrine,
        'Family_Smoking_Habit': smoking,
        'Social_Assistance_Recipient': social_assist,
        'Health_Insurance_Status': health_insurance,
        'Data_Source': data_source,
        'Monitoring_Type': monitoring_type,
    }

    input_df = pd.DataFrame([raw_input])

    # Lengkapi kolom fitur yang belum terisi dari form (diisi NaN, akan ditangani imputer)
    for col in numeric_features + categorical_features:
        if col not in input_df.columns:
            input_df[col] = np.nan

    # --- Preprocessing (identik dengan pipeline saat training) ---
    input_num = numeric_imputer.transform(input_df[numeric_features])
    input_cat = categorical_imputer.transform(input_df[categorical_features])

    input_cat_encoded = encoder.transform(input_cat)
    input_num_scaled = scaler.transform(input_num)

    input_final = np.hstack([input_num_scaled, input_cat_encoded])

    # --- Prediksi ---
    proba = model.predict_proba(input_final)[0, 1]
    prediction = int(proba >= best_threshold)

    st.divider()
    st.subheader("Hasil Prediksi")
    st.metric("Probabilitas Risiko Stunting", f"{proba*100:.1f}%")

    if prediction == 1:
        st.error("⚠️ **Risiko Stunting Terdeteksi**")
        st.write("**Rekomendasi:** Segera lakukan verifikasi pengukuran ulang dan rujuk ke tenaga kesehatan / Puskesmas untuk pemeriksaan lebih lanjut.")
    else:
        st.success("✅ **Risiko Stunting Rendah**")
        st.write("**Rekomendasi:** Lanjutkan pemantauan rutin melalui Posyandu.")

    st.caption("Catatan: Hasil ini adalah prediksi model berbasis data, bukan diagnosis medis. Keputusan klinis tetap harus melalui tenaga kesehatan.")

> 💡 **Insight:** Perhatikan bagian preprocessing di `app.py` — urutannya harus **PERSIS SAMA** dengan urutan yang dipakai di notebook saat training (imputer → encoder untuk kategorikal, scaler untuk numerik → digabung `np.hstack`). Kalau urutannya beda atau kolomnya tidak lengkap, hasil prediksi bisa salah tanpa error yang jelas (silent bug) — ini kesalahan paling umum saat deployment model ML.

## Cara Menjalankan Aplikasi

**Opsi A — Coba dulu langsung di Colab (sementara, untuk demo cepat):**

Jalankan cell di bawah ini. Colab tidak bisa langsung membuka Streamlit, jadi kita pakai `localtunnel` untuk membuat URL publik sementara ke server Streamlit yang jalan di Colab.

In [ ]:
!pip install streamlit --quiet
!npm install -g localtunnel --silent

import subprocess, time

# Jalankan Streamlit di background
streamlit_process = subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.headless', 'true'])
time.sleep(5)

# Tampilkan IP publik Colab (dipakai sebagai password localtunnel jika diminta)
!curl -s https://loca.lt/mytunnelpassword

# Buat tunnel publik ke port 8501
!npx localtunnel --port 8501

Klik link yang muncul (bentuknya `https://xxxx.loca.lt`), lalu masukkan IP di atas sebagai password saat diminta oleh halaman localtunnel. Ini **hanya untuk demo/testing** — link akan mati begitu runtime Colab berhenti.

**Opsi B — Deployment permanen (direkomendasikan untuk dipakai Yayasan sungguhan):**

1. Buat repository baru di GitHub, upload 3 hal ini ke dalamnya:
   - `app.py` (sudah dibuat di atas)
   - `requirements.txt` (dibuat di cell berikutnya)
   - folder `model_artifacts/` (berisi 6 file `.pkl` yang sudah disimpan di Tahap 14)
2. Buka [streamlit.io/cloud](https://streamlit.io/cloud) → Sign in pakai akun GitHub → **New app**
3. Pilih repository tadi, branch, dan file utama `app.py`
4. Klik **Deploy** — Streamlit Cloud otomatis install `requirements.txt` dan menjalankan aplikasinya
5. Setelah selesai, kamu dapat URL publik permanen (format `https://namaapp.streamlit.app`) yang bisa dibuka siapa saja, kapan saja, tanpa perlu Colab menyala

Alternatif lain selain Streamlit Cloud: **Hugging Face Spaces** (juga gratis, caranya mirip) atau deploy sebagai REST API pakai **FastAPI** di **Render**/**Railway** kalau butuh diintegrasikan ke aplikasi lain (bukan cuma form web).

In [ ]:
%%writefile requirements.txt
streamlit
pandas
numpy
scikit-learn
joblib
imbalanced-learn

## Ringkasan Bagian Modeling & Deployment

**Modeling:**
- Melatih & membandingkan 4 algoritma (Logistic Regression, Decision Tree, Random Forest, Gradient Boosting) pada data hasil SMOTENC
- Evaluasi memakai Precision, Recall, F1-Score, ROC-AUC, dan PR-AUC — bukan cuma Accuracy — karena target imbalanced
- Model terbaik dipilih otomatis berdasarkan F1-Score, lalu threshold keputusan dioptimalkan (bukan asal pakai 0.5)
- Model + seluruh objek preprocessing (imputer, encoder, scaler) disimpan sebagai artifact `.pkl` agar bisa dipakai ulang tanpa perlu retraining

**Deployment:**
- Dibungkus jadi aplikasi web Streamlit (`app.py`) dengan form input yang mudah dipakai oleh non-teknis (kader Posyandu/tim Yayasan)
- Pipeline preprocessing di aplikasi dibuat identik dengan pipeline saat training agar prediksi konsisten
- Disediakan 2 opsi: demo cepat via Colab + localtunnel, dan deployment permanen via Streamlit Community Cloud

**Catatan penting:** karena dataset yang dipakai untuk training di notebook ini adalah data dummy/simulasi, model yang dihasilkan **belum siap dipakai untuk keputusan nyata**. Seluruh pipeline (dari cleaning, Z-score WHO, sampai modeling & deployment) sudah divalidasi berjalan dengan benar dan **siap diterapkan ulang ke data asli Yayasan** begitu tahap cleaning data mentahnya selesai.